# 01. Covariate Encoder and Model Forward Walkthrough

This notebook explains `src/models/covariate_encoding.py` with a real Replogle batch, then runs one forward pass through the loaded checkpoint model.

The goal is not to train or sample. The goal is to see what the model receives and produces internally:

1. Build the same Replogle sampling config used by the inference notebook.
2. Build the datamodule and load the checkpoint-backed `PlModel`.
3. Pull one real batch from the current dataset.
4. Run `CovEncoder` branch by branch: perturbation, cell type, batch.
5. Compare the manual branch concat/projection with `model._encode_covariates()`.
6. Run one direct Cross-DiT forward pass and inspect output shapes.

## 0. What `CovEncoder` Is Doing

`CovEncoder` turns discrete experimental metadata into a dense condition vector.

For the Replogle setting in this notebook:

- `cov_pert`: perturbation/gene id, shape roughly `[batch, cell_set]`
- `cov_celltype`: cell line/type id, shape roughly `[batch, cell_set]`
- `cov_batch`: experimental batch id, shape roughly `[batch, cell_set]`

Each branch maps ids to vectors of `hidden_dim=128`, then the vectors are concatenated and projected to `output_dim=2000`:

```text
pert id ----> pert_encoder     ---- 128 dim --+
cell id ----> celltype_encoder ---- 128 dim --+--> concat 384 dim --> Linear --> 2000 dim
batch id ---> batch_encoder    ---- 128 dim --+
```

That final 2000-d vector is called `batch_emb` in the rest of the model. Cross-DiT then embeds it into transformer hidden size and uses it together with the diffusion timestep.

## 1. Imports and Local Paths

In [1]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path("/home/lwf/PerturbDiff").resolve()
if not (PROJECT_ROOT / "src" / "models" / "covariate_encoding.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "models" / "covariate_encoding.py").exists(), PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
CKPT_PATH = PROJECT_ROOT / "checkpoints" / DATASET_NAME / "from_scratch_replogle.ckpt"
RUN_DIR = PROJECT_ROOT / "runs" / "replogle_model_forward_walkthrough"
WANDB_DIR = PROJECT_ROOT / "wandb"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PERTURB_ROOT:", PERTURB_ROOT)
print("CKPT_PATH:", CKPT_PATH)
assert PERTURB_ROOT.exists(), "Run example/replogle/00_download_data.ipynb first."
assert CKPT_PATH.exists(), "Run example/replogle/00_download_data.ipynb first."

PROJECT_ROOT: /home/lwf/PerturbDiff
PERTURB_ROOT: /home/lwf/PerturbDiff/data/replogle/perturb_data
CKPT_PATH: /home/lwf/PerturbDiff/checkpoints/replogle/from_scratch_replogle.ckpt


In [2]:
import numpy as np
import torch
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
from pytorch_lightning.utilities import move_data_to_device

from src.common.utils import setup_loggings
from src.apps.sampling.sampling_setup import (
    build_sampling_datamodule,
    load_sampling_model,
    populate_covariate_cfg,
)
from src.apps.sampling.sampling_utils import setup_device
from src.apps.sampling.sampling_generation_helpers import build_gene_embedding_cache, build_self_condition

## 2. Compose the Hydra Config

This matches the Replogle inference settings. We use the existing sampling helpers because they already handle checkpoint loading and datamodule setup.

In [3]:
overrides = [
    "path=trixie_path",
    f"path.tmp_dir={PERTURB_ROOT}",
    f"path.diffusion.save_dir={RUN_DIR}",
    f"path.wandb.logging_dir={WANDB_DIR}",
    f"model_checkpoint_path={CKPT_PATH}",
    "data=replogle_finetune",
    "data.normalize_counts=10",
    "data.pad_length=2000",
    "data.embed_key=X_hvg",
    "model.hidden_num=[2000,512]",
    "model.input_dim=2000",
    "optimization.micro_batch_size=16",
    "data.use_cell_set=8",
    "cov_encoding=trixie_onehot",
    "cov_encoding.batch_encoding=onehot",
    "cov_encoding.celltype_encoding=llm",
    "cov_encoding.replogle_gene_encoding=genept",
    "model.p_drop_control=0",
    "data.keep_control_cell=false",
    "sampling.use_ddim=true",
    "sampling.num_sampled_batches=1",
    "trainer.devices=[0]",
    "trainer.use_distributed_sampler=false",
    "lightning.logger._target_=pytorch_lightning.loggers.logger.DummyLogger",
    "~lightning.logger.project",
    "~lightning.logger.save_dir",
    "~lightning.logger.name",
]

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "configs"), version_base=None):
    cfg = compose(config_name="rawdata_diffusion_sampling", overrides=overrides)

OmegaConf.resolve(cfg)
logger = setup_loggings(cfg)

print("data_name:", cfg.data.data_name)
print("dataset_path:", cfg.data.dataset_path)
print("cov encodings:")
print("  pert_encoding:", cfg.cov_encoding.pert_encoding)
print("  replogle_gene_encoding:", cfg.cov_encoding.replogle_gene_encoding)
print("  celltype_encoding:", cfg.cov_encoding.celltype_encoding)
print("  batch_encoding:", cfg.cov_encoding.batch_encoding)

05/19/2026 16:16:16 - INFO - src.common.utils -  Configuration args
05/19/2026 16:16:16 - INFO - src.common.utils -  {'run_name': 'sampling', 'save_dir_path': None, 'model_checkpoint_path': '/home/lwf/PerturbDiff/checkpoints/replogle/from_scratch_replogle.ckpt', 'device': 'auto', 'sampling': {'sample_unperturbed': False, 'num_sampled_batches': 1, 'batch_size': 16, 'use_ddim': True, 'clip_denoised': True, 'progress': True, 'guidance_strength': 1.0, 'start_time': 100, 'nw': 0.5, 'start_guide_steps': 500, 'eta': 0.0, 'output_format': 'npz', 'output_dir': None}, 'trainer': {'accelerator': 'gpu', 'devices': [0], 'num_nodes': 1, 'use_distributed_sampler': False, 'default_root_dir': '/home/lwf/PerturbDiff/runs/replogle_model_forward_walkthrough', 'enable_checkpointing': True, 'enable_model_summary': True, 'enable_progress_bar': True, 'deterministic': True, 'check_val_every_n_epoch': 1, 'val_check_interval': 1.0, 'log_every_n_steps': 20, 'num_sanity_val_steps': 0, 'max_epochs': None, 'max_step

## 3. Build Datamodule, Populate Covariate Config, Load Model

`populate_covariate_cfg()` fills runtime values that `CovEncoder` needs: cardinalities and label-to-index dictionaries.

In [4]:
torch.manual_seed(cfg.optimization.seed)

t0 = time.time()
datamodule = build_sampling_datamodule(cfg, logger)
populate_covariate_cfg(cfg, datamodule)
print(f"datamodule setup took {time.time() - t0:.2f}s")

print("num_pert:", cfg.cov_encoding.num_pert)
print("num_celltype:", cfg.cov_encoding.num_celltype)
print("num_batch:", cfg.cov_encoding.num_batch)
print("example perturbation dict items:", list(cfg.cov_encoding.pert_dict.items())[:5])
print("cell_type_dict:", cfg.cov_encoding.cell_type_dict)
print("example batch dict items:", list(cfg.cov_encoding.batch_dict.items())[:5])

05/19/2026 16:16:21 - INFO - src.common.utils -  Calling pretraining data module setup ...


Processing meta data ...: 1it [00:00,  2.52it/s]

05/19/2026 16:16:21 - INFO - src.common.utils -  #perturbation: 2024, #cell type: 4, #batch: 56
datamodule setup took 0.51s
num_pert: 2024
num_celltype: 4
num_batch: 56
example perturbation dict items: [(np.str_('AAAS'), 0), (np.str_('AAMP'), 1), (np.str_('AAR2'), 2), (np.str_('AARS'), 3), (np.str_('AARS2'), 4)]
cell_type_dict: {np.str_('hepg2'): 0, np.str_('jurkat'): 1, np.str_('k562'): 2, np.str_('rpe1'): 3}
example batch dict items: [('replogle_1', 0), ('replogle_10', 1), ('replogle_11', 2), ('replogle_12', 3), ('replogle_13', 4)]


In [5]:
t0 = time.time()
model = load_sampling_model(cfg, logger, datamodule)
device = setup_device(cfg, logger)
model = model.to(device)
model.eval()
print(f"model load took {time.time() - t0:.2f}s")
print("device:", device)
print("PlModel contains:", type(model.cov_encoder).__name__, type(model.model).__name__, type(model.diffusion).__name__)
print("model_cfg.hidden_num:", model.model_cfg.hidden_num)
print("model.model.input_size after x/self-conditioning concat:", model.model.input_size)
print("model.model.hidden_size:", model.model.hidden_size)

05/19/2026 16:16:31 - INFO - src.common.utils -  Using device: cuda
model load took 6.19s
device: cuda
PlModel contains: CovEncoder Cross_DiT GaussianDiffusion
model_cfg.hidden_num: [2000, 512, 500, 500]
model.model.input_size after x/self-conditioning concat: 4000
model.model.hidden_size: 512


## 4. Pull One Real Batch

The sampling code uses `datamodule.val_dataloader()[1]`, the test split loader. Here we inspect the raw batch before any model call.

In [6]:
t0 = time.time()
datamodule.setup_dataset()
dataloader = datamodule.val_dataloader()[1]
batch_cpu = next(iter(dataloader))
print(f"dataset + first batch took {time.time() - t0:.2f}s")

def describe(x):
    if torch.is_tensor(x):
        return f"Tensor shape={tuple(x.shape)} dtype={x.dtype} device={x.device}"
    if isinstance(x, np.ndarray):
        return f"ndarray shape={x.shape} dtype={x.dtype}"
    if isinstance(x, (list, tuple)):
        first_type = type(x[0]).__name__ if len(x) else "empty"
        return f"{type(x).__name__} len={len(x)} first_type={first_type}"
    return f"{type(x).__name__}: {str(x)[:120]}"

for key in sorted(batch_cpu.keys()):
    print(f"{key:>18}: {describe(batch_cpu[key])}")

05/19/2026 16:16:49 - INFO - src.common.utils -  Calling pretraining data module setup ...


Splitting into train/validation/test data (only for indices) ...: 0it [00:00, ?it/s]

05/19/2026 16:16:56 - INFO - src.common.utils -  Processed train: 611710
05/19/2026 16:16:56 - INFO - src.common.utils -  Processed validation: 43990
05/19/2026 16:16:56 - INFO - src.common.utils -  Processed test: 66043


Splitting into train/validation/test data (only for indices) ...: 1it [00:06,  6.62s/it]

05/19/2026 16:16:56 - INFO - src.common.utils -  >>>>> stage: train
05/19/2026 16:16:56 - INFO - src.common.utils -  For dataset replogle (different split), control cells: 39165, perturb cells: 572545


05/19/2026 16:16:57 - INFO - src.common.utils -  Shared gene count in data replogle: 		---- (2000 / 2000; original: 6642)
05/19/2026 16:16:57 - INFO - src.common.utils -  >>>>> stage: validation
05/19/2026 16:16:57 - INFO - src.common.utils -  For dataset replogle (different split), control cells: 39165, perturb cells: 4825
05/19/2026 16:16:58 - INFO - src.common.utils -  Shared gene count in data replogle: 		---- (2000 / 2000; original: 6642)
05/19/2026 16:16:58 - INFO - src.common.utils -  >>>>> stage: test
05/19/2026 16:16:58 - INFO - src.common.utils -  For dataset replogle (different split), control cells: 39165, perturb cells: 26878
05/19/2026 16:16:59 - INFO - src.common.utils -  Shared gene count in data replogle: 		---- (2000 / 2000; original: 6642)
05/19/2026 16:16:59 - WARNING - src.data.sampler -  this sampler should only be used when doing sampling
05/19/2026 16:16:59 - WARNING - src.data.sampler -  this sampler is ignoring the drop_last argument
05/19/2026 16:16:59 - WARN

## 5. The Three Raw Covariate Inputs

These three tensors are exactly what `PlModel._encode_covariates()` passes into `CovEncoder.forward()`.

In [7]:
batch = move_data_to_device(batch_cpu, device)

cov_pert = batch["cov_pert"]
cov_celltype = batch["cov_celltype"]
cov_batch = batch["cov_batch"]

print("cov_pert:", describe(cov_pert), "min/max:", int(cov_pert.min()), int(cov_pert.max()))
print("cov_celltype:", describe(cov_celltype), "unique:", torch.unique(cov_celltype).detach().cpu().tolist())
print("cov_batch:", describe(cov_batch), "unique sample:", torch.unique(cov_batch).detach().cpu().tolist()[:10])
print("is_padded_list:", describe(batch["is_padded_list"]), "valid:", int((~batch["is_padded_list"].bool()).sum()))
print("pert_emb:", describe(batch["pert_emb"]))
print("cont_emb:", describe(batch["cont_emb"]))

cov_pert: Tensor shape=(2, 8) dtype=torch.int64 device=cuda:0 min/max: 8 8
cov_celltype: Tensor shape=(2, 8) dtype=torch.int64 device=cuda:0 unique: [0]
cov_batch: Tensor shape=(2, 8) dtype=torch.int64 device=cuda:0 unique sample: [0, 33]
is_padded_list: Tensor shape=(2, 8) dtype=torch.int64 device=cuda:0 valid: 3
pert_emb: Tensor shape=(2, 8, 2000) dtype=torch.float32 device=cuda:0
cont_emb: Tensor shape=(2, 8, 2000) dtype=torch.float32 device=cuda:0


## 6. Inspect the `CovEncoder` Modules

The checkpoint config controls which branches exist. In this run, Replogle perturbations use `genept`, cell types use external LLM embeddings, and batch uses one-hot embeddings.

In [ ]:
cov_encoder = model.cov_encoder
cov_cfg = model.cov_encoding_cfg

print("cov_cfg:")
print("  pert_encoding:", cov_cfg.pert_encoding)
print("  drug_encoding:", cov_cfg.drug_encoding)
print("  replogle_gene_encoding:", cov_cfg.replogle_gene_encoding)
print("  celltype_encoding:", cov_cfg.celltype_encoding)
print("  batch_encoding:", cov_cfg.batch_encoding)
print("  hidden_dim:", cov_cfg.hidden_dim)
print("  output_dim:", cov_cfg.output_dim)

print("\nmodules:")
for name, module in cov_encoder.named_children():
    print(f"  {name}: {module}")

## 7. Run Each `CovEncoder` Branch Manually

This cell mirrors `CovEncoder.forward()` but stores every intermediate tensor so we can inspect it.

Important branch behavior:

- `replogle_gene_encoding == "genept"`: `pert_encoder(cov_pert)` uses pretrained gene embeddings projected to 128 dim.
- `pert_encoding == "non"`: would ignore perturbation ids and use zero ids.
- otherwise one-hot perturbation uses `pert_input + 1` because index 0 is reserved.
- `celltype_encoder(cov_celltype)` is either one-hot or pretrained LLM embedding projected to 128 dim.
- `batch_encoder(cov_batch)` is optional.

In [ ]:
branch_outputs = []

with torch.no_grad():
    if cov_cfg.drug_encoding == "chemberta_cls":
        pert_branch = cov_encoder.drug_encoder(cov_pert)
        dose_branch = cov_encoder.dose_encoder(cov_encoder.dose_indices[cov_pert])
        branch_outputs.extend([pert_branch, dose_branch])
        print("drug_branch:", describe(pert_branch))
        print("dose_branch:", describe(dose_branch))
    elif cov_cfg.replogle_gene_encoding == "genept":
        pert_branch = cov_encoder.pert_encoder(cov_pert)
        branch_outputs.append(pert_branch)
        print("replogle genept perturbation branch:", describe(pert_branch))
    elif cov_cfg.pert_encoding == "non":
        pert_branch = cov_encoder.pert_encoder(torch.zeros_like(cov_pert, dtype=cov_pert.dtype))
        branch_outputs.append(pert_branch)
        print("non perturbation branch:", describe(pert_branch))
    else:
        pert_branch = cov_encoder.pert_encoder(cov_pert + 1)
        branch_outputs.append(pert_branch)
        print("onehot/esm2 perturbation branch:", describe(pert_branch))

    celltype_branch = cov_encoder.celltype_encoder(cov_celltype)
    branch_outputs.append(celltype_branch)
    print("celltype branch:", describe(celltype_branch))

    if hasattr(cov_encoder, "batch_encoder"):
        batch_branch = cov_encoder.batch_encoder(cov_batch)
        branch_outputs.append(batch_branch)
        print("batch branch:", describe(batch_branch))

    concat_repr = torch.cat(branch_outputs, dim=-1)
    manual_batch_emb = cov_encoder.transform(concat_repr)

print("concat_repr:", describe(concat_repr))
print("manual_batch_emb after transform:", describe(manual_batch_emb))
print("manual_batch_emb stats:", float(manual_batch_emb.mean()), float(manual_batch_emb.std()))

## 8. Compare with `PlModel._encode_covariates()`

`PlModel._encode_covariates()` is just a thin wrapper around the same `CovEncoder.forward()` call.

In [ ]:
with torch.no_grad():
    wrapped_batch_emb = model._encode_covariates(batch)

max_abs_diff = (manual_batch_emb - wrapped_batch_emb).abs().max().item()
print("wrapped_batch_emb:", describe(wrapped_batch_emb))
print("max absolute difference from manual path:", max_abs_diff)

batch["batch_emb"] = wrapped_batch_emb

## 9. Build the Cross-DiT `self_condition` Dict

During sampling and training, Cross-DiT receives a dictionary with condition tensors. The most important field from `CovEncoder` is `batch_emb`.

In [ ]:
with torch.no_grad():
    gene_emb = build_gene_embedding_cache(model, batch, device)
    self_condition = build_self_condition(cfg, model, batch, gene_emb)

print("self_condition:")
for key, value in self_condition.items():
    print(f"  {key}: {describe(value)}")

## 10. Run One Direct Cross-DiT Forward Pass

Cross-DiT's input embedder expects `hidden_num[0] * 2` features because the diffusion code concatenates the current input with a self-conditioning estimate.

For this inspection pass, we use zeros as the self-conditioning estimate:

```python
x_input = cat([pert_emb, zeros_like(pert_emb)], dim=-1)
x_control_input = cat([cont_emb, zeros_like(cont_emb)], dim=-1)
```

This is not a full DDIM sampling loop. It is one raw model forward call at a fixed diffusion timestep.

In [ ]:
x_start = batch["pert_emb"]
control_start = batch["cont_emb"]
x_selfcond0 = torch.zeros_like(x_start)
control_selfcond0 = torch.zeros_like(control_start)

x_input = torch.cat([x_start, x_selfcond0], dim=-1)
x_control_input = torch.cat([control_start, control_selfcond0], dim=-1)

t_raw = torch.full((x_start.shape[0],), 100, device=device, dtype=torch.long)
t_model = model.diffusion._scale_timesteps(t_raw).unsqueeze(1)

print("x_start:", describe(x_start))
print("x_input after concat:", describe(x_input))
print("x_control_input after concat:", describe(x_control_input))
print("t_raw:", describe(t_raw), t_raw[:5].detach().cpu().tolist())
print("t_model:", describe(t_model))

In [ ]:
with torch.no_grad():
    forward_output = model.model(
        x_input,
        x_control_input,
        t_model,
        self_condition=self_condition,
    )

print("forward output keys:", sorted(forward_output.keys()))
for key, value in forward_output.items():
    print(f"{key:>26}: {describe(value)}")

print("\nOutput stats:")
print("x mean/std/min/max:", float(forward_output["x"].mean()), float(forward_output["x"].std()), float(forward_output["x"].min()), float(forward_output["x"].max()))
print("x_control mean/std/min/max:", float(forward_output["x_control"].mean()), float(forward_output["x_control"].std()), float(forward_output["x_control"].min()), float(forward_output["x_control"].max()))

## 11. Optional: Run the Diffusion Wrapper's Model Call

The diffusion code normally calls `model.diffusion.get_model_output(...)`, which does the same Cross-DiT call after scaling timesteps and concatenating self-conditioning. This cell runs that wrapper for comparison.

In [ ]:
RUN_DIFFUSION_WRAPPER_FORWARD = False

if RUN_DIFFUSION_WRAPPER_FORWARD:
    with torch.no_grad():
        wrapper_output = model.diffusion.get_model_output(
            model.model,
            x_t=x_start,
            control_input_t=control_start,
            t=t_raw,
            self_condition=self_condition,
            model_kwargs={},
        )
    print("wrapper output keys:", sorted(wrapper_output.keys()))
    for key, value in wrapper_output.items():
        print(f"{key:>26}: {describe(value)}")

## 12. Reading the Shapes

For a Replogle batch with `data.use_cell_set=8`, many tensors are shaped like:

```text
[outer batch, cell_set, genes_or_features]
```

Typical shape story:

- `cov_pert`, `cov_celltype`, `cov_batch`: `[B, S]`
- each covariate branch output: `[B, S, 128]`
- concatenated covariate representation: `[B, S, 384]` for three branches
- `batch_emb`: `[B, S, 2000]`
- Cross-DiT input after self-conditioning concat: `[B, S, 4000]`
- Cross-DiT output `x`: `[B, S, 2000]`

Conceptually, `CovEncoder` answers: "what perturbation/cell/batch condition is this cell-set under?" Cross-DiT then uses that condition plus timestep plus control expression to predict the perturbed expression.